# GPU 記憶體優化實戰指南

GPU 記憶體通常是深度學習訓練的最大限制。本教學將教你如何：

- 💾 **最大化記憶體利用率**：訓練更大的模型或批次
- 🔧 **診斷記憶體問題**：OOM、記憶體洩漏
- ⚡ **優化技巧集錦**：梯度檢查點、混合精度、激活重計算

## 內容概覽
1. 記憶體使用基礎
2. 梯度檢查點 (Gradient Checkpointing)
3. 梯度累積 (Gradient Accumulation)
4. 激活重計算
5. 其他優化技巧

In [ ]:
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"設備: {device}")

if torch.cuda.is_available():
    print(f"總記憶體: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 1. 梯度檢查點 (Gradient Checkpointing)

**原理**: 訓練時不保存所有中間激活，而是在反向傳播時重新計算
**權衡**: 犧牲 20-30% 速度，節省 50-80% 記憶體

In [ ]:
class CheckpointedModel(nn.Module):
    """使用梯度檢查點的模型"""
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(1024, 1024) for _ in range(20)])
    
    def forward(self, x, use_checkpoint=True):
        for layer in self.layers:
            if use_checkpoint:
                # 使用梯度檢查點
                x = checkpoint(lambda x: torch.relu(layer(x)), x, use_reentrant=False)
            else:
                x = torch.relu(layer(x))
        return x

if torch.cuda.is_available():
    model = CheckpointedModel().to(device)
    x = torch.randn(128, 1024, device=device)
    
    # 不使用檢查點
    torch.cuda.reset_peak_memory_stats()
    output = model(x, use_checkpoint=False)
    output.sum().backward()
    mem_no_cp = torch.cuda.max_memory_allocated() / 1024**2
    
    # 使用檢查點
    torch.cuda.reset_peak_memory_stats()
    output = model(x, use_checkpoint=True)
    output.sum().backward()
    mem_with_cp = torch.cuda.max_memory_allocated() / 1024**2
    
    print(f"不使用檢查點: {mem_no_cp:.2f} MB")
    print(f"使用檢查點: {mem_with_cp:.2f} MB")
    print(f"節省: {(1 - mem_with_cp/mem_no_cp)*100:.1f}%")

## 2. 梯度累積

**原理**: 小批次多次前向/反向傳播，累積梯度後再更新
**好處**: 等效大批次訓練，但記憶體使用小

In [ ]:
def train_with_gradient_accumulation():
    """梯度累積訓練示例"""
    model = models.resnet18().to(device)
    optimizer = torch.optim.Adam(model.parameters())
    
    # 設置
    small_batch_size = 16  # 實際批次
    accumulation_steps = 4  # 累積步數
    effective_batch_size = small_batch_size * accumulation_steps  # 等效批次 = 64
    
    print(f"\n等效批次大小: {effective_batch_size}")
    print(f"實際批次大小: {small_batch_size}")
    print(f"累積步數: {accumulation_steps}")
    
    for i in range(accumulation_steps):
        # 小批次數據
        data = torch.randn(small_batch_size, 3, 224, 224, device=device)
        target = torch.randint(0, 1000, (small_batch_size,), device=device)
        
        # 前向傳播
        output = model(data)
        loss = nn.functional.cross_entropy(output, target)
        
        # 梯度縮放（平均）
        loss = loss / accumulation_steps
        loss.backward()
        
        print(f"  Step {i+1}/{accumulation_steps} 完成")
    
    # 累積完成後才更新
    optimizer.step()
    optimizer.zero_grad()
    
    print("✓ 梯度累積完成，參數已更新")

if torch.cuda.is_available():
    train_with_gradient_accumulation()

## 3. 記憶體優化技巧速查

### 3.1 及時釋放變量

In [ ]:
# ❌ 不好：保留不必要的變量
def bad_memory_usage():
    all_losses = []
    for i in range(100):
        loss = model(data)
        all_losses.append(loss)  # 保留計算圖！

# ✓ 好：只保存標量值
def good_memory_usage():
    all_losses = []
    for i in range(100):
        loss = model(data)
        all_losses.append(loss.item())  # 只保存數值

print("""
記憶體優化快速提示：

1. 使用 .detach() 分離不需要梯度的張量
2. 使用 .item() 提取標量值
3. 使用 del 刪除大型中間變量
4. torch.no_grad() 包裹推理代碼
5. 定期調用 torch.cuda.empty_cache()
""")

### 3.2 混合精度訓練

In [ ]:
from torch.cuda.amp import autocast, GradScaler

if torch.cuda.is_available():
    model = models.resnet50().to(device)
    
    # FP32 基線
    torch.cuda.reset_peak_memory_stats()
    x = torch.randn(64, 3, 224, 224, device=device)
    _ = model(x)
    mem_fp32 = torch.cuda.max_memory_allocated() / 1024**2
    
    # FP16 混合精度
    torch.cuda.reset_peak_memory_stats()
    with autocast():
        _ = model(x)
    mem_fp16 = torch.cuda.max_memory_allocated() / 1024**2
    
    print(f"\nFP32: {mem_fp32:.2f} MB")
    print(f"AMP (FP16): {mem_fp16:.2f} MB")
    print(f"節省: {(1 - mem_fp16/mem_fp32)*100:.1f}%")
    print("\n提示: 混合精度可以節省約 40-50% 記憶體")

## 4. OOM 問題診斷

當遇到 `CUDA out of memory` 錯誤時：

In [ ]:
def diagnose_oom():
    """
    OOM 診斷檢查清單
    """
    print("""
=== CUDA OOM 診斷步驟 ===

1. 檢查批次大小
   → 減小 batch_size
   → 使用梯度累積

2. 啟用混合精度
   → from torch.cuda.amp import autocast
   → with autocast(): ...

3. 使用梯度檢查點
   → from torch.utils.checkpoint import checkpoint
   → checkpoint(fn, *args)

4. 減少模型層數或寬度
   → 使用更小的模型架構

5. 清理記憶體
   → torch.cuda.empty_cache()
   → del large_tensors

6. 分析記憶體使用
   → print(torch.cuda.memory_summary())
    """)

diagnose_oom()

## 5. 完整優化示例

In [ ]:
def memory_efficient_training_template():
    """
    記憶體高效訓練模板
    """
    print("""
# 記憶體高效訓練模板

from torch.cuda.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint

# 設置
model = YourModel().to(device)
optimizer = torch.optim.AdamW(model.parameters())
scaler = GradScaler()  # AMP

# 梯度累積設置
accumulation_steps = 4

for epoch in range(num_epochs):
    for i, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)
        
        # 混合精度前向傳播
        with autocast():
            output = model(data)
            loss = criterion(output, target) / accumulation_steps
        
        # 縮放梯度並反向傳播
        scaler.scale(loss).backward()
        
        # 梯度累積完成後更新
        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        # 定期清理（可選）
        if i % 100 == 0:
            torch.cuda.empty_cache()
    """)

memory_efficient_training_template()

## 總結

### 記憶體優化工具箱

| 技術 | 記憶體節省 | 速度影響 | 難度 | 推薦度 |
|------|-----------|---------|------|--------|
| 混合精度 (AMP) | 40-50% | +30-100% | ⭐ | ⭐⭐⭐⭐⭐ |
| 梯度累積 | 0% | 0% | ⭐ | ⭐⭐⭐⭐⭐ |
| 梯度檢查點 | 50-80% | -20-30% | ⭐⭐ | ⭐⭐⭐⭐ |
| 減小批次 | 線性 | 負面 | ⭐ | ⭐⭐⭐ |
| 模型並行 | 50%+ | -10-20% | ⭐⭐⭐⭐ | ⭐⭐⭐ |

### 推薦優化順序

1. **首先啟用 AMP** - 最大收益，最小代價
2. **使用梯度累積** - 等效大批次，無額外成本
3. **考慮梯度檢查點** - 如果仍然 OOM
4. **最後才減小模型** - 可能影響精度